In [ ]:
!pip install datasets

In [ ]:
# Split the dataset into val and test
from sklearn.model_selection import train_test_split

from datasets import load_dataset
import os
import json
from google.colab import drive
from sklearn.model_selection import train_test_split


# Output paths
root_dir = "/mmstar/data"
image_root_dir = os.path.join(root_dir, "images")

json_output_path = os.path.join(root_dir, "problems.json")
split_output_path = os.path.join(root_dir, "pid_splits.json")
os.makedirs(image_root_dir, exist_ok=True)

# Load MMStar val split
dataset = load_dataset("Lin-Chen/MMStar")["val"]
#filter for the science & technology
filtered_dataset = [example for example in dataset if example.get("category") == "science & technology"]
dataset_list = list(filtered_dataset)

# Assuming 'dataset' is the dataset loaded from MMStar
val_dataset, test_dataset = train_test_split(dataset_list, test_size=0.2, random_state=42)

# Initialize lists to store question IDs for each split
val_ids = []
test_ids = []

# Initialize output dictionaries
problems = {}
answer_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
# Helper to get choices
def extract_choices(meta_info):
    keys = ['A', 'B', 'C', 'D']
    return [meta_info.get(k, f"Option {k}") for k in keys] if isinstance(meta_info, dict) else ["A", "B", "C", "D"]


# val dataset
for idx, example in enumerate(val_dataset):
    qid = str(idx)
    val_ids.append(qid)

    # Save image
    image_subdir = os.path.join(image_root_dir, qid)
    os.makedirs(image_subdir, exist_ok=True)
    image_path = os.path.join(image_subdir, "image.png")
    try:
        example["image"].save(image_path)
    except Exception as e:
        print(f"[!] Error saving image {qid}: {e}")
        continue

    choices = extract_choices(example.get("meta_info", {}))
    answer = example["answer"]
    answer_index = answer_map.get(answer, -1)
    if answer_index == -1:
          print(f"[!] Error: Answer for question {qid} ({answer}) not found in answer_map")

    problems[qid] = {
        "question": example["question"],
        "choices": choices,
        "answer": answer_index,
        "hint": "",
        "image": "image.png",
        "task": "closed choice",
        "grade": "grade2",
        "subject": example.get("category", "vision"),
        "topic": example.get("l2_category", ""),
        "category": example.get("category", "vision"),
        "skill": "",
        "lecture": "",
        "solution": "",
        "split": "val"  # Set split to val
    }

# Test dataset
for idx, example in enumerate(test_dataset):
    qid = str(idx + len(val_dataset))  # Ensure unique question IDs
    test_ids.append(qid)

    # Save image
    image_subdir = os.path.join(image_root_dir, qid)
    os.makedirs(image_subdir, exist_ok=True)
    image_path = os.path.join(image_subdir, "image.png")
    try:
        example["image"].save(image_path)
    except Exception as e:
        print(f"[!] Error saving image {qid}: {e}")
        continue

    choices = extract_choices(example.get("meta_info", {}))
    answer = example["answer"]
    answer_index = answer_map.get(answer, -1)
    if answer_index == -1:
          print(f"[!] Error: Answer for question {qid} ({answer}) not found in answer_map")
    problems[qid] = {
        "question": example["question"],
        "choices": choices,
        "answer": answer_index,
        "hint": "",
        "image": "image.png",
        "task": "closed choice",
        "grade": "grade2",
        "subject": example.get("category", "vision"),
        "topic": example.get("l2_category", ""),
        "category": example.get("category", "vision"),
        "skill": "",
        "lecture": "",
        "solution": "",
        "split": "test" 
    }


# Save problems.json
with open(json_output_path, "w") as f:
    json.dump(problems, f, indent=2)

# Save pid_splits.json
pid_splits = {
    "val": val_ids,
    "test": test_ids  
}
with open(split_output_path, "w") as f:
    json.dump(pid_splits, f, indent=2)

print(f" Saved problems.json and pid_splits.json to: {root_dir}")
